Note: Get a free AEMET API key at --> https://opendata.aemet.es/centrodedescargas/altaUsuario

# Imports

In [1]:
import os
from pathlib import Path
import requests

import pandas as pd

from dotenv import load_dotenv

load_dotenv()

AEMET_API_KEY = os.getenv("AEMET_API_KEY")

# Parameters

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
INPUT_DATA_DIR = DATA_DIR / "input"

In [3]:
stations = {
    "Madrid": "3195",
    "Barcelona": "0201D",
    "Valencia": "8416",
    "Sevilla": "5783",
    "Bilbao": "1082",
    "Zaragoza": "9434",
}

# Get Data

In [4]:
weather_dfs = []

start_date = "2026-01-01T00:00:00UTC"
end_date = "2026-06-30T23:59:59UTC"

for city, station_id in stations.items():

    endpoint_aemet = (
        "https://opendata.aemet.es/opendata/api/"
        f"valores/climatologicos/diarios/datos/"
        f"fechaini/{start_date}/"
        f"fechafin/{end_date}/"
        f"estacion/{station_id}"
    )

    response_aemet = requests.get(
        endpoint_aemet,
        params={"api_key": AEMET_API_KEY},
        timeout=30,
    )

    response_aemet.raise_for_status()

    metadata_aemet = response_aemet.json()

    data_url = metadata_aemet["datos"]

    weather_response = requests.get(
        data_url,
        timeout=30,
    )

    weather_response.raise_for_status()

    weather_data = weather_response.json()

    df_city = pd.DataFrame(weather_data)

    df_city["city"] = city

    weather_dfs.append(df_city)

# Preprocess

In [5]:
df_weather_all = pd.concat(
    weather_dfs,
    ignore_index=True,
)

In [9]:
df_weather_all = df_weather_all[
    [
        "fecha",
        "tmed",
        "tmin",
        "tmax",
        "prec",
        "velmedia",
    ]
].copy()

In [ ]:
df_weather_all['velmedia'] = df_weather_all['velmedia'].fillna(0)

AttributeError: 'Series' object has no attribute 'to_float'

In [16]:
df_weather_all.groupby('fecha').agg({'tmed': 'mean', 
                                     'tmin': 'mean', 
                                     'tmax': 'mean', 
                                     'prec': 'mean', 
                                     'velmedia': 'mean'})

TypeError: dtype 'str' does not support operation 'mean'

In [13]:
df_weather_all[df_weather_all['fecha']=='2026-01-01']

,fecha,tmed,tmin,tmax,prec,velmedia
0,2026-01-01,"2,8","0,3","5,4","0,0","1,1"
179,2026-01-01,"10,2","6,4","13,9","0,0","2,2"
283,2026-01-01,"10,6","5,4","15,8","0,0",NaN
464,2026-01-01,"11,0","7,7","14,3","0,1","3,6"
645,2026-01-01,"6,6","1,6","11,7","0,0","3,3"
826,2026-01-01,"1,5","0,1","2,9","0,0","2,2"


# Save Data

In [11]:
weather_output_path = (
    INPUT_DATA_DIR / "02_meteo.csv"
)

df_weather_all.to_csv(
    weather_output_path,
    index=False,
)